In [ ]:
import $ivy.`org.apache.spark::spark-sql:4.1.4`

In [ ]:
org.apache.spark.sql.catalyst.encoders.OuterScopes.addOuterScope(this)
case class Estudiante(estudiante: String, nota: Double, estado: String)

In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("VerificacionScala")
  .master("local[*]")
  .config("spark.jars.ivy", "C:/SparkLocal/tools/cache/ivy2")
  .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

import spark.implicits._
println(s"Versión de Spark en Scala: ${spark.version}")

// 1. RDD + lambda + map
def sc = spark.sparkContext
val rddRes = sc.parallelize(Seq(1, 2, 3, 4, 5)).map(x => x * 2).filter(_ > 4).collect()
println("RDD map + lambda: " + rddRes.mkString(", "))

// 2. DataFrame + UDF
val df = Seq(("Ana", 9.5), ("Luis", 6.2), ("Sofía", 9.8)).toDF("estudiante", "nota")
val clasificarUdf = udf((nota: Double) => if (nota >= 9.0) "Sobresaliente" else "Aprobado")
val dfResultado = df.withColumn("estado", clasificarUdf($"nota"))
dfResultado.show()

// 3. Dataset tipado
val ds = dfResultado.as[Estudiante]
ds.filter(_.nota >= 9.0).show()

// 4. Parquet (verifica winutils.exe y hadoop.dll)
val rutaParquet = "C:/SparkLocal/workspace/notebooks/tmp_test_scala.parquet"
ds.write.mode("overwrite").parquet(rutaParquet)
spark.read.parquet(rutaParquet).show()

spark.stop()